# Mission 1 / 2 · 음성 파인튜닝 (Colab GPU)로컬 CPU 베이스라인은 MFCC/F0 통계 + 얕은 분류기. 이 노트북은 wav2vec2 를 파인튜닝한다.**중요**: 원본은 8 kHz 협대역이고 wav2vec2 는 16 kHz 광대역 사전학습이다. 단순 업샘플만으로도동작하지만 대역 불일치로 성능이 깎인다 (Sivaraman & Khoury, Odyssey'20). 업샘플은 GPU 쪽에서수행하고, 업로드는 8 kHz 원본으로 해 용량을 절반으로 줄인다.`python -m src.preprocess.pack_for_colab --what audio` 로 만든 npz 를 `MyDrive/dcc/` 에 올릴 것.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')DIR = '/content/drive/MyDrive/dcc'!pip -q install "transformers>=4.44" torchaudio scikit-learnimport torch, numpy as np, torchaudioprint('cuda', torch.cuda.is_available())

In [ ]:
MISSION = 'm2'      # 'm1' = 성별(통화 단위), 'm2' = 화자 역할(발화 단위). 이 값만 바꾸면 전환된다.def load(split):    """번들을 읽어 조각 리스트로 되돌린다.    번들은 모든 오디오를 하나의 긴 int16 배열(audio)로 이어붙이고 경계를 offsets 에 담았다.    조각을 개별 배열로 저장하면 npz 안에 수만 개의 엔트리가 생겨 로딩이 매우 느려지기 때문이다.    offsets 는 길이 n+1 이고 i 번째 조각은 audio[offsets[i]:offsets[i+1]] 이다.    """    d = np.load(f'{DIR}/{MISSION}_audio_{split}.npz')    a, off, y = d['audio'], d['offsets'], d['y']    segs = [a[off[i]:off[i+1]] for i in range(len(off)-1)]    # overlap: 다른 화자와 시간이 겹치는가 (M2 진단용)    # call: 어느 통화에서 나온 조각인가 (통화 단위 분석용)    extra = {k: d[k] for k in ('overlap', 'call') if k in d}    return segs, y.astype(np.int64), extratr_x, tr_y, _ = load('train')va_x, va_y, va_extra = load('val')print(f'train {len(tr_x):,}  val {len(va_x):,}  양성비율 {tr_y.mean():.3f}')if 'overlap' in va_extra:    print(f'val 중첩 비율 {va_extra["overlap"].mean():.3f}')

In [ ]:
from torch.utils.data import Dataset, DataLoaderfrom transformers import AutoFeatureExtractor, AutoModelForAudioClassificationMODEL = 'facebook/wav2vec2-base'SEC = 3.0            # 모든 조각을 이 길이로 맞춘다. 배치로 묶으려면 길이가 같아야 하고,                     # 3초는 발화 길이 p95(5.4초)와 중앙값(1.4초) 사이의 절충값이다.SR_IN, SR_OUT = 8000, 16000# 8 kHz -> 16 kHz 업샘플. wav2vec2 는 16 kHz 로 사전학습돼 있어 입력 샘플레이트를 맞춰야 한다.# 업샘플이 없는 정보를 만들어내지는 못한다 (원본은 여전히 0.3-3.4 kHz 전화 대역).# 대역 불일치로 성능이 깎이는 건 알려진 문제이며, 개선하려면 대역확장(BWE)이 필요하다.resamp = torchaudio.transforms.Resample(SR_IN, SR_OUT)N = int(SEC * SR_OUT)      # 고정 입력 길이 = 48,000 샘플fe = AutoFeatureExtractor.from_pretrained(MODEL)# num_labels=2 -> M1 은 (여성, 남성), M2 는 (119대원, 신고자). 둘 다 이진 분류라 코드를 공유한다.model = AutoModelForAudioClassification.from_pretrained(MODEL, num_labels=2).cuda()class ADS(Dataset):    """int16 조각 -> 16 kHz 고정 길이 float 파형."""    def __init__(self, segs, y):        self.s, self.y = segs, y    def __len__(self):        return len(self.s)    def __getitem__(self, i):        # int16 (-32768..32767) -> float [-1, 1). 번들을 int16 로 저장해 용량을 절반으로 줄였다.        w = torch.from_numpy(self.s[i].astype(np.float32) / 32768.0)        w = resamp(w)        if len(w) < N:            # 짧은 발화는 뒤를 0 으로 채운다. 맞장구("예", "네")가 0.5초 미만이라 이 경우가 흔하다.            w = torch.nn.functional.pad(w, (0, N - len(w)))        elif len(w) > N:            # 긴 발화는 가운데를 자른다. 발화 시작/끝은 묵음이나 겹침이 섞이기 쉬워 중앙이 안전하다.            o = (len(w) - N) // 2            w = w[o:o+N]        return w, self.y[i]def collate(b):    """배치를 쌓고 파형별로 표준화한다.    wav2vec2 의 feature extractor 는 zero-mean/unit-variance 입력을 가정한다(do_normalize=True).    여기서는 조각마다 개별 정규화하므로, 통화별 녹음 음량 차이가 모델에 노출되지 않는다.    M2 에서는 이 점을 유의할 것 — 음량/채널 차이가 신고자 vs 대원의 실제 단서이기 때문에,    정규화가 오히려 유용한 신호를 지울 수 있다. 정규화를 끈 버전과 비교해볼 가치가 있다.    """    x = torch.stack([i[0] for i in b])    x = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-7)    return x, torch.tensor([i[1] for i in b])# batch_size 는 T4 16GB 기준. OOM 이 나면 16 -> 8 로 줄일 것.tr_dl = DataLoader(ADS(tr_x, tr_y), batch_size=16, shuffle=True, collate_fn=collate, num_workers=2)va_dl = DataLoader(ADS(va_x, va_y), batch_size=32, collate_fn=collate, num_workers=2)print(f'배치당 입력 shape: {next(iter(tr_dl))[0].shape}  (batch, 48000 샘플 = 3초 @16kHz)')

In [ ]:
from torch.optim import AdamW# lr 3e-5 는 wav2vec2 파인튜닝의 관례적 범위(1e-5 ~ 5e-5). 더 크면 사전학습 표현이 무너진다.opt = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)# 혼합정밀(AMP). T4 에서 메모리를 절반으로 줄이고 속도를 약 2배 올린다.# GradScaler 는 fp16 언더플로로 gradient 가 0 이 되는 것을 막아준다.scaler = torch.amp.GradScaler('cuda')@torch.no_grad()def evaluate():    """Validation 전체 예측. argmax 이므로 임계값 0.5 고정과 동일하다."""    model.eval()    P = []    for x, _ in va_dl:        with torch.amp.autocast('cuda'):            P.append(model(input_values=x.cuda()).logits.float().argmax(-1).cpu().numpy())    return np.concatenate(P)# Colab 무료 티어는 세션이 끊길 수 있으므로 epoch 마다 평가하고 결과를 남긴다.# 18만 세그먼트 x 1 epoch 은 T4 에서 1~2시간이 걸린다.for ep in range(2):    model.train()    tot = 0    for i, (x, y) in enumerate(tr_dl):        opt.zero_grad(set_to_none=True)        with torch.amp.autocast('cuda'):            # labels 를 넘기면 HF 모델이 CrossEntropyLoss 를 내부에서 계산한다.            out = model(input_values=x.cuda(), labels=y.cuda())        scaler.scale(out.loss).backward()        scaler.step(opt)        scaler.update()        tot += out.loss.item()        if (i+1) % 200 == 0:            print(f'ep{ep+1} {i+1}/{len(tr_dl)} loss {tot/(i+1):.4f}', flush=True)    p = evaluate()    acc = (p == va_y).mean()    print(f'== epoch {ep+1}  val acc {acc:.4f}   (CPU 베이스라인: M1 0.9434 / M2 0.8799)')    # M2 의 핵심 진단: 발화의 37% 가 다른 화자와 시간적으로 겹친다.    # 로컬 베이스라인에서 비중첩 0.8944 vs 중첩 0.8550 으로 3.9p 차이가 났다.    # 이 격차가 줄어드는지가 wav2vec2 도입의 실질적 판단 기준이다.    if 'overlap' in va_extra:        ov = va_extra['overlap']        print(f'   비중첩 {(p[ov==0]==va_y[ov==0]).mean():.4f} / 중첩 {(p[ov==1]==va_y[ov==1]).mean():.4f}')

In [ ]:
import matplotlib.pyplot as plt, matplotlib# Colab 기본 이미지에는 한글 폰트가 없어 축 라벨이 두부(□)로 깨진다. 나눔고딕을 설치해 등록한다.!apt-get -qq install fonts-nanum > /dev/nullmatplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')matplotlib.rc('font', family='NanumGothic')matplotlib.rc('axes', unicode_minus=False)      # 한글 폰트 사용 시 음수 기호가 깨지는 것 방지p = evaluate()labels = ['여성 F', '남성 M'] if MISSION == 'm1' else ['119대원', '신고자']# 혼동행렬을 [[TN, FP], [FN, TP]] 순서로 만든다. 행=정답, 열=예측.cm = np.array([[((va_y == i) & (p == j)).sum() for j in (0, 1)] for i in (0, 1)], dtype=float)pct = cm / np.maximum(cm.sum(1, keepdims=True), 1)      # 행 정규화 = 클래스별 재현율fig, ax = plt.subplots(figsize=(4.2, 3.8))ax.imshow(pct, cmap='Blues', vmin=0, vmax=1)ax.set_xticks([0, 1], labels)ax.set_yticks([0, 1], labels)ax.set_xlabel('예측')ax.set_ylabel('정답')ax.set_title(f'{MISSION.upper()} wav2vec2  acc={(p==va_y).mean():.4f}')for i in range(2):    for j in range(2):        # 진한 셀에는 흰 글씨, 옅은 셀에는 검은 글씨로 대비를 확보한다.        ax.text(j, i, f'{int(cm[i,j]):,}\n{pct[i,j]*100:.1f}%', ha='center', va='center',                color='white' if pct[i, j] > 0.55 else '#16202B')plt.tight_layout(); plt.show()# 체크포인트는 Drive 에 저장한다. Colab 런타임 디스크는 연결이 끊기면 사라진다.torch.save({'state_dict': model.state_dict(), 'model_name': MODEL, 'sec': SEC},           f'{DIR}/ckpt/{MISSION}.pt')print('저장 완료:', f'{DIR}/ckpt/{MISSION}.pt')